# D19 S3 A1 - IoU, Confidence Filtering, and NMS


In [1]:
import numpy as np

boxes = [[0, 0, 10, 10], [1, 1, 10, 10], [20, 20, 30, 30], [21, 21, 30, 30]]
scores = [0.90, 0.80, 0.70, 0.60]


In [2]:
def iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    inter_w = max(0, min(ax2, bx2) - max(ax1, bx1))
    inter_h = max(0, min(ay2, by2) - max(ay1, by1))
    intersection = inter_w * inter_h

    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - intersection

    if union == 0:
        return 0.0
    return float(intersection / union)


def filter_by_confidence(boxes, scores, thresh):
    return [i for i, score in enumerate(scores) if score >= thresh]


def nms(boxes, scores, iou_thresh):
    remaining = sorted(range(len(boxes)), key=lambda i: scores[i], reverse=True)
    keep = []

    while remaining:
        best = remaining.pop(0)
        keep.append(best)
        remaining = [
            i for i in remaining
            if iou(boxes[best], boxes[i]) <= iou_thresh
        ]

    return keep


## Verification


In [3]:
print(round(iou([0,0,10,10], [0,0,10,10]), 4))
print(round(iou([0,0,10,10], [20,20,30,30]), 4))
print(round(iou([0,0,10,10], [5,5,15,15]), 4))
print(filter_by_confidence(boxes, scores, 0.5))
print(nms(boxes, scores, 0.5))
print(nms(boxes, scores, 0.95))


1.0
0.0
0.1429
[0, 1, 2, 3]
[0, 2]
[0, 1, 2, 3]


## Autograder-style checks


In [4]:
assert np.isclose(iou([0,0,10,10], [0,0,10,10]), 1.0)
assert np.isclose(iou([0,0,10,10], [20,20,30,30]), 0.0)
assert np.isclose(iou([0,0,10,10], [5,5,15,15]), 25/175)
assert filter_by_confidence(boxes, scores, 0.7) == [0, 1, 2]
assert nms(boxes, scores, 0.5) == [0, 2]
assert nms(boxes, scores, 0.95) == [0, 1, 2, 3]
outer = [0, 0, 10, 10]
inner = [2, 2, 4, 4]
assert np.isclose(iou(outer, inner), 0.04)
assert np.isclose(iou(inner, outer), 0.04)
cluster_boxes = [[1,1,10,10], [0,0,10,10], [20,20,30,30]]
cluster_scores = [0.8, 0.9, 0.7]
assert nms(cluster_boxes, cluster_scores, 0.5) == [1, 2]
assert iou([0,0,0,0], [0,0,0,0]) == 0.0
print("All 8 autograder-style checks passed.")


All 8 autograder-style checks passed.
